In [ ]:
# Example 1: Default Proxy is the C-accelerated cext implementation.
# Verify the C extension is loaded (not the pure-Python fallback).
import lazy_object_proxy
from lazy_object_proxy import Proxy

print(f"Proxy class: {Proxy}")
print(f"Proxy module: {Proxy.__module__}")

# The cext module should be importable.
from lazy_object_proxy import cext
print(f"cext module: {cext}")
print(f"cext.Proxy: {cext.Proxy}")

assert Proxy is cext.Proxy, "Default Proxy should be the C-accelerated one"

In [ ]:
# Example 2: Basic lazy evaluation - the factory is only called when accessed.
from lazy_object_proxy import Proxy

call_count = {"n": 0}

def expensive_factory():
    call_count["n"] += 1
    return [1, 2, 3, 4, 5]

proxy = Proxy(expensive_factory)

# Factory should not have been called yet.
assert call_count["n"] == 0, "Factory should be lazy"

# Accessing the proxy triggers the factory.
assert len(proxy) == 5
assert call_count["n"] == 1

# Subsequent accesses reuse the cached value.
assert proxy[0] == 1
assert sum(proxy) == 15
assert call_count["n"] == 1, "Factory should be called exactly once"
print(f"Factory invocations: {call_count['n']}")


In [ ]:
# Example 3: Proxy transparency - it acts like the wrapped object.
from lazy_object_proxy import Proxy

class Greeter:
    def __init__(self, name):
        self.name = name
    def hello(self):
        return f"hello {self.name}"

p = Proxy(lambda: Greeter("world"))

# Attribute access and method calls transparently delegate.
assert p.name == "world"
assert p.hello() == "hello world"

# isinstance checks also pass through to the wrapped type.
assert isinstance(p, Greeter)

# Type of the proxy itself is still Proxy.
print(f"type(p) = {type(p).__name__}")
print(f"p.hello() = {p.hello()}")


In [ ]:
# Example 4: Pure-Python slots / simple fallback implementations work too.
from lazy_object_proxy import slots, simple

# Both fallback implementations expose a Proxy class.
slots_p = slots.Proxy(lambda: {"a": 1, "b": 2})
simple_p = simple.Proxy(lambda: {"a": 1, "b": 2})

assert slots_p["a"] == 1
assert simple_p["b"] == 2
assert dict(slots_p) == dict(simple_p) == {"a": 1, "b": 2}

print(f"slots.Proxy: {slots.Proxy.__module__}.{slots.Proxy.__name__}")
print(f"simple.Proxy: {simple.Proxy.__module__}.{simple.Proxy.__name__}")


In [ ]:
# Example 5: Arithmetic and comparison operators are forwarded to the wrapped object.
from lazy_object_proxy import Proxy

n = Proxy(lambda: 42)
s = Proxy(lambda: "abc")

# Numeric ops on the proxy yield real values.
assert n + 8 == 50
assert n * 2 == 84
assert n > 10
assert n == 42

# String ops too.
assert s + "def" == "abcdef"
assert s.upper() == "ABC"
assert len(s) == 3

# repr of the proxy unwraps too.
print(f"repr(n) = {repr(n)}")
print(f"repr(s) = {repr(s)}")

assert int(n) == 42 and str(s) == "abc"
